# Time Operators

Streaming data is infinite. Kafi Streams is an in-memory stream processor. Memory is never infinite.

So of course you need *time operators* that effectively clean up memory so that your Kafi Streams processing pipeline has constant, not ever-growing memory usage.

Freeing memory of timed out data is implemented as [expiry](#expiry) in Kafi Streams.

The time operators also allow you to implement the [*time windows*](#windows) as e.g. in Kafka Streams, such as [tumbling](#tumbling), [hopping](#hopping), [cumulative](#cumulative), [sliding](#sliding) and [session](#session) windows. And moreover, Kafi Streams enables you to build arbitrary [new types of time windows](#custom) as well.

## Overview

[Preparation](#prep)

* [Expiry](#expiry)
  * [expire()](#expire-operator)
* [Time windows](#windows)
  * [Time Windows = expire + group + aggregate](#expire_group_aggregate)
  * [Tumbling windows](#tumbling)
    * [expire_tumbling()](#expire_tumbling-operator)
    * [group_by_agg_tumbling()](#group_by_agg_tumbling-operator)
  * [Hopping windows](#hopping)
    * [expire_hopping()](#expire_hopping-operator)
    * [group_by_agg_hopping()](#group_by_agg_hopping-operator)
  * [Cumulative windows](#cumulative)
    * [expire_cumulative()](#expire_cumulative-operator)
    * [group_by_agg_cumulative()](#group_by_agg_cumulative-operator)
  * [Sliding windows](#sliding)
    * [expire_sliding()](#expire_sliding-operator)
    * [group_by_agg_sliding()](#group_by_agg_sliding-operator)
  * [Session windows](#session)
    * [expire_session()](#expire_session-operator)
    * [group_by_agg_session()](#group_by_agg_session-operator)
  * [Triggers and Custom windows](#custom)
    * [Session/Threshold windows](#threshold)

---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [44]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, "../")
sys.path.insert(1, "../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator, OrderGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()
order_generator = OrderGenerator()

click_source_str = "clicks"
customer_source_str = "customers"
order_source_str = "orders"
sink_str = "sink"

#

def run(built_tn):
    sink_m_list = []
    for i in range(100):
        # 1. Generate new data.
        click_m_list = click_generator.generate(100)
        customer_m_list = customer_generator.generate(100)

        # 2. Push the new data to the topology + incrementally process the new data + get the resulting changes.
        sink_str_m_list_dict = built_tn.process({click_source_str: click_m_list, customer_source_str: customer_m_list})
        m_list = sink_str_m_list_dict[sink_str]

        # 3. Print out the size of the pydbsp state.
        sys.stdout.write(f"\rStep: {i + 1}, Memory: {built_tn.get_state_size() / 1024}KB")

        # 4. Add the changes to the output list.
        sink_m_list += m_list

    print()
    print(len(sink_m_list))
    print(sink_m_list[-10:])

def process(built_tn, customer_id, price, ts, w=1):
    m = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    #
    sink_str_r_list_dict = built_tn.process({order_source_str: [(m, w)]})
    r_list = sink_str_r_list_dict[sink_str]
    #
    print("Triggers:")
    for r in r_list:
        print(r)



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Please also note that when we re-use the same example over and over again to illustrate how the operators work, we always mark the important new parts as follows:
```python
    # <------------------------------>
    ...important new parts...
    # <------------------------------>
```

---
<a id="expiry"></a>
## Expiry

Expiry is the central concept in Kafi Streams for freeing memory of timed out data.

Thanks to pydbsp, Kafi Streams can implement expiry natively, without having to bolt on any kind of mechanism on top.

Essentially, expiry has to be defined only once for each source at the beginning of the Kafi Streams topology. All the stateful operators downstream do not need any special handling - they are automatically cleaned up by the expired records percolating through the topology, one by one.

We need an example. Let us recollect the example from the [Quickstart](../quickstart.ipynb) using the `TopologyNode` class (see [Architecture](../architecture.ipynb))


In [ ]:
click_source_str = "clicks"
customer_source_str = "customers"

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

sink_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
    .sink(sink_str)
)

built_tn = Tn.build(sink_tn)


And then, let us throw data at it and see how the global state size of the topology grows:

In [ ]:
built_tn.reset()
for _ in range(3):
    run(built_tn)

This is of course not sustainable. The memory usage grows unboundedly and Kafi Streams gets slower and slower (in this case, mostly caused by the join).

It's time to introduce the `expire()` operator.

<a id="expire-operator"></a>
### expire()

Expires individual records so that they can be purged from memory.

```
expire(ts_fun, expiry_fun, project_fun=lambda r_ts_tuple: r_ts_tuple[0], **kwargs)
```
* `ts_fun: r -> ts`: timestamp function - gets an input record and returns a timestamp
* `expiry_fun: ts -> ts`: expiry function - gets a timestamp and returns the corresponding expiry timestamp specifying when the record shall expire
* `project_fun: tuple(r, ts) -> r`: projection function - gets a pair of a record and its expiry timestamp and returns another record. Default: `lambda r_ts_tuple: r_ts_tuple[0]`

Let's add this to our topology.


In [ ]:
click_source_str = "clicks"
customer_source_str = "customers"

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    # <------------------------------>
    .expire(ts_fun=lambda r: r["ts"],
            expiry_fun=lambda ts: ts + click_generator.ts_step_int * 1000)
    # <------------------------------>
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

sink_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
    .sink(sink_str)
)

built_tn = Tn.build(sink_tn)


Here, we use the `expiry()` operator to:
1. Select the `ts` field from each record,
2. and then set the expiry to the selected timestamp plus `1000` times the timestamp step size of the click generator.

When you look closer at the topology, you can observe multiple some of the properties of expiry in Kafi Streams:
* It is defined once at the top of the topology, before any stateful operator (`map` and `filter` are stateless).
* The stateful operators (`distinct` and `join_equi`) below in the topology do not need to know anything about the expiry.

Ok. Let's see this in action. Will be able to rein in the memory consumption?

In [ ]:
for _ in range(5):
    run(built_tn)

It works! Constant, flat memory usage! No processing slowdown!

Why? Because we used `expire()` to time out the transactional data (=the clicks). After a short while, the memory consumption of the master data (=the customers) becomes constant as well because it is limited (the generator only generates up to 100 customers in the example).

---
<a id="windows"></a>
## Time windows

In the previous section, we learnt how we can keep Kafi Streams' memory usage at check. It was only remotely related to time windows in the classical stream processing sense: under the covers, the `expire()` operator works akin to a "sliding window" in classical stream processing.

This section is about "real" stream processing time windows.

You'll see that we devised a novel formulation of them inside DBSP that allows us to build all the time window types from classical stream processing.

But it doesn't stop there - Kafi Streams is so flexible that you can easily build your own custom time windows.


<a id="expire_group_aggregate"></a>
### Time Windows = expire + group + aggregate

What is a time window really? You can think of time windows in stream processing as consisting of two ingredients:
* **Expiry**: Time windows have a start and an end. Events *expire* after the end of a time window so that they can be cleaned up.
* **Group By + Aggregate**: The actual "time window" is a set of events *grouped* by time and some other key (e.g. a customer ID) and *aggregated*.

Now this is very theoretical. Let's pick the simplest time window - the *tumbling window* and see how all this theory plays out in practice.

In [ ]:
# <------------------------------>

def ts_fun(r):
    return r["ts"]

size_int = click_generator.ts_step_int * 1000

# <------------------------------>

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    
    # <------------------------------>
    
    .expire_tumbling(ts_fun=ts_fun,
                     size_int=size_int)
    
    # <------------------------------>
    
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

joined_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]})
)

# <------------------------------>

sink_tn = (
    joined_tn
    .group_by_agg_tumbling(
        ts_fun=ts_fun,
        size_int=size_int,
        key_fun=lambda r: {"customer_id": r["customer_id"], "name": r["name"]},
        agg_fun=lambda agg_r, r: {"clicks": agg_r["clicks"] + 1,
                                  "view_times": agg_r["view_times"] + [r["view_time"]],
                                  "total_view_time": agg_r["total_view_time"] + r["view_time"]},
        agg_initial_any={"clicks": 0, "view_times": [], "total_view_time": 0},
        project_fun=lambda key_any, agg_r: {"customer_id": key_any["customer_id"],
                                            "name": key_any["name"],
                                            "clicks": agg_r["clicks"],
                                            "view_times": agg_r["view_times"],
                                            "total_view_time": agg_r["total_view_time"]})
    .sink(sink_str)
)

# <------------------------------>

built_tn = Tn.build(sink_tn)


What do we do here?

* **expire**: At the top of the topology, we specify the expiry of the incoming clicks using the `expire_tumbling()` operator and set the tumbling window size to `tumbling_size_int = click_generator.ts_step_int * 1000`.
* **group + aggregate**: After the join of clicks and customers, we create the tumbling window using the `group_by_agg_tumbling()` operator: We group by `customer_id` and `name`, and aggregate the clicks for that customer in that time window:
  * `clicks` the number of clicks of the customer
  * `view_times` the list of view times of the customer
  * `total_view_time` the total view time of the customer

Let's run this.

In [ ]:
run(built_tn)

This was your first time window in Kafi Streams in action!

In the following sections, we double down on the individual built-in window types of Kafi Streams and explain their API in detail.

<a id="tumbling"></a>
### Tumbling Windows

In Kafi Streams, the two operators required to set up a tumbling window are `expire_tumbling` and `group_by_agg_tumbling`.

<a id="expire_tumbling-operator"></a>
#### expire_tumbling()

Syntactic sugar for `expire()` for record expiry in the context of tumbling windows.

```
expire_tumbling(ts_fun, size_int, allowed_lateness_int=0, **kwargs)
```
* `ts_fun: r -> ts` timestamp function - gets an input record and returns a timestamp
* `size_int` the size (in milliseconds) of the tumbling window
* `allowed_lateness_int` allowed lateness - adds a time (in milliseconds) to the expiry time to accommodate late arriving records


<a id="group_by_agg_tumbling-operator"></a>
#### group_by_agg_tumbling()

Augmented `group_by_agg()` operator for creating tumbling time windows by grouping by + aggregating. Implicitly also groups by the tumbling time windows and triggers the emission of aggregated tumbling time windows.

```
group_by_agg_tumbling(ts_fun, size_int, key_fun, agg_fun, agg_initial_any, project_fun, trigger_fun=lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1], trigger_project_fun=lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]}, trigger_positive_only_bool=True, **kwargs)
```

* `ts_fun: r -> ts`: timestamp function - gets an input record and returns a timestamp
* `size_int`: the size (in milliseconds) of the window
* `key_fun: r -> any`: the selection function for the key for the grouping
* `agg_fun: agg_any, value_any -> any`: the aggregation function; gets the result of the aggregation so far (`agg_any`) and the next selected value to aggregate (`value_any`), and returns the updated aggregation
* `agg_initial_any`: the initial value of the aggregation
* `project_fun: key_any, agg_any -> r`: the projection function; gets both the selected key and the aggregation result for that key and returns the output (=projection) of the group by + aggregation
* `trigger_fun: tuple(r, end_ts), latest_ts -> bool` the trigger function; gets a pair of a record and the window end timestamp and the latest timestamp of the input stream and returns a bool. `True` for triggering the emission of the output, `False` for not yet triggering it. Default: `lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1]` where `r_end_ts_tuple[1] = end_ts`- i.e., trigger if the latest timestamp of the input stream is greater than or equal to the window end timestamp `end_ts`.
* `trigger_project_fun: tuple(r, end_ts) -> r`: the trigger projection function; gets a pair of a record and the window end timestamp and returns a record. Default: `lambda r_end_ts_tuple: {**r_end_ts_int_tuple[0], "window_end": r_end_ts_int_tuple[1]}` i.e., add the window end timestamp to the record.
* `trigger_positive_only_bool`: trigger only updates with positive weights or also zero or negative ones. Default: `True`

This looks scary at first. But for most use cases, only these parameters are obligatory:
* `ts_fun` - same as in `expire_tumbling()`
* `size_int` - same as in `expire_tumbling()`
* `key_fun` - as in `group_by_agg()`
* `agg_fun` - as in `group_by_agg()`
* `agg_initial_any` - as in `group_by_agg()`
* `project_fun` - as in `group_by_agg()`

The `trigger_` parameters should only be required for advanced use cases. They control the emission of time windows based on a triggering mechanism. Their defaults should suffice for most uses cases. We'll show a use case for them when we discuss custom time windows at the end of this notebook.

Note that contrary to the basic `group_by_agg()` operator, there is no `value_fun`. This is because the `value_fun` in `group_by_agg_tumbling` is always the identity function since we assume that in 99% of the use cases, you would want to create time windows around entire records.

Ok. After so many parameters, we need to see them in action. Here is an example, this time not about clicks and customers, but just orders to keep it simple:


In [35]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

size_int = 100
allowed_lateness_int = size_int * 2
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    # 1. Select customer_id, price and ts from the value.
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    # 2. Expire with window size order_generator.ts_step_int * 100 and allowed_lateness = window_size * 2,  
    .expire_tumbling(lambda r: r["ts"], size_int, allowed_lateness_int)
    # 3. Deduplicate.
    .distinct()
)
#
# 4. Set up the tumbling window: group by customer ID, count the orders, sum up the prices of the orders and get the last timestamp of the window.
sink_tn = order_tn.group_by_agg_tumbling(
    ts_fun=lambda r: r["ts"],
    size_int=size_int,
    key_fun=lambda r: r["customer_id"],
            agg_fun=lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                      "total_price": agg_r["total_price"] + r["price"],
                                      "last_ts": max(agg_r["last_ts"], r["ts"])},
            agg_initial_any={"orders": 0, "total_price": 0, "last_ts": 0},
            project_fun=lambda key_any, agg_r: {"customer_id": key_any,
                                                "orders": agg_r["orders"],
                                                "total_price": agg_r["total_price"],
                                                "last_ts": agg_r["last_ts"]},
    trigger_positive_only_bool=False
).sink(sink_str)
#
_ = built_tn = Tn.build(sink_tn)

What do we do?
1. `map()`: Select customer_id, price and ts from the value.
2. `expire_tumbling()`: Expire with window size `100` and `allowed_lateness` = `size_int * 2 = 200`.  
3. `distinct()`: Deduplicate.
4. `group_by_agg_tumbling()`: Set up the tumbling window: group by customer ID, count the orders sum up the prices of the orders and get the last timestamp of the window.

Next, we illustrate how the tumbling window works by processing some example data - one by one, in baby steps.

In the illustrations:
* time proceeds from top to bottom, (starting with `0`)
* the start and end times of the time windows are marked by small grey circles.
* the latest timestamp of the input after the respective step is written at the top 
* new events coming in a step are indicated a blue frame
* old events have a grey frame
* the triggered outputs in the sink are indicated by green color



#### Step 1

In step 1, first, the first order arrives from customer 1 at timestamp 10:

```mermaid
flowchart LR
    subgraph x_axis ["Latest timestamp = 10"]
        direction TB
        0(("0")) e1@-.-> 10
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#1565c0,stroke-width:6px
```

The first tumbling window covers the period `[0, 100)`, i.e., from `0` until `99`.

As the latest timestamp is not yet beyond the end of the first tumbling window, no output is triggered. Why?

Because the default `trigger_fun` is defined as `lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1]` where `r_end_ts_tuple[1] = end_ts = 100`. At this point, `latest_ts = 10 >= 100` evaluates to `False` and consequently, no output is triggered.

Let's see this happening for real:

In [36]:
process(built_tn, customer_id=1, price=100, ts=10, w=1)


Triggers:
[]


#### Step 2

In baby step 2, the second order arrives from customer 1 at timestamp 50:

```mermaid
flowchart LR
    subgraph x_axis ["Latest timestamp = 50"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}"}
    style 50 fill:none,stroke:#1565c0,stroke-width:6px
```

The latest timestamp still not beyond the end of the first tumbling window `[0, 100)`, no output is triggered:

In [37]:
process(built_tn, customer_id=1, price=200, ts=50, w=1)


Triggers:
[]


#### Step 3

In step 3, an order from customer arrives shortly after the end of the first tumbling window:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 105"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 105
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 2,\n&quot;total_price&quot;: 300,\n&quot;last_ts&quot;: 50\n&quot;window_end&quot;: 100}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    space1[ ]
    style space1 fill:none,stroke:none
    105 --- space1
    linkStyle 4 stroke:none
    105 Link@== Triggers ==> Output
    linkStyle 5 stroke:#00bb00,stroke-width:3px
```

Now the latest timestamp of the input stream *is* beyond the end of the first tumbling window.

Hence, the aggregation of the two orders from customer 1 that came in between `[0, 100)` is triggered, because `latest_ts = 105 >= 100` evaluates to `True` in `trigger_fun`:

In [39]:
process(built_tn, customer_id=2, price=50, ts=105, w=1)

Triggers:
[]


#### Step 4

Step 4 shows the effect of a retraction coming in (weight = `-1`) as the order from customer 1 at timestamp `50` is canceled:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 105"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 105
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s style='color:red;'>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#bb0000,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 100,\n&quot;last_ts&quot;: 10\n&quot;window_end&quot;: 100}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    50 Link@== Triggers ==> Output
    linkStyle 4 stroke:#00bb00,stroke-width:3px;
```

The correction still comes in early enough (as in still within the `200` ms of `allowed_lateness`: `latest_ts = 105 - 50 = 55 < 200`). As a result, the corresponding correction of the tumbling window `[0, 100)` is triggered.

In [40]:
process(built_tn, customer_id=1, price=200, ts=50, w=-1)


Triggers:
[{'customer_id': 1, 'orders': 1, 'total_price': 100, 'last_ts': 10, 'window_end': 100}]


#### Step 5

An order from customer 3 arrives out of order at timestamp `101`:

```mermaid
flowchart LR
    subgraph x_axis ["Latest timestamp = 105"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#1565c0,stroke-width:6px
```

The tumbling window `[0, 100)` for customer 1 hasn't changed. The order from customer 3 lands in the tumbling windows for `[100, 200)`, but as the latest timestamp is still at `105`, it is also not triggered yet:


In [41]:
process(built_tn, customer_id=3, price=400, ts=101, w=1)

Triggers:
[]


#### Step 6

Another order from customer 3 arrives at timestamp `150`:

```mermaid
flowchart LR
    subgraph x_axis ["Latest timestamp = 150"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105 e6@-.-> 150
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#1565c0,stroke-width:6px
```

We are still not beyond the second tumbling window `[100, 200)`. No output.

In [42]:
process(built_tn, customer_id=3, price=200, ts=150, w=1)


Triggers:
[]


#### Step 7

Customer 1 creates another order at timestamp 210:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 210"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105 e6@-.-> 150 e7@-.-> 200(("200")) e8@-.-> 210
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    210@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 210}"}
    style 210 fill:none,stroke:#1565c0,stroke-width:6px

    Output1@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 50\n&quot;last_ts&quot;: 105,\n&quot;window_end&quot;: 200}"}
    style Output1 fill:none,stroke:#00bb00,stroke-width:6px
    210 Link@== Triggers ==> Output1
    linkStyle 8 stroke:#00bb00,stroke-width:3px;

    Output2@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;orders&quot;: 2,\n&quot;total_price&quot;: 600,\n&quot;last_ts&quot;: 150,\n&quot;window_end&quot;: 200}"}
    style Output2 fill:none,stroke:#00bb00,stroke-width:6px
    210 Link@== Triggers ==> Output2
    linkStyle 9 stroke:#00bb00,stroke-width:3px;
```

This pushes the latest timestamp to `210`, i.e., beyond `[100, 200)`. Two aggregations for tumbling window `[100, 200)` are triggered: For customer 2 and 3.

In [45]:
process(built_tn, customer_id=1, price=50, ts=210, w=1)

Triggers:


#### Step 8

An order from customer 2 arrives at timestamp `330`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 330"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105 e6@-.-> 150 e7@-.-> 200(("200")) e8@-.-> 210 e9@-.-> 300(("300")) e10@-.-> 330
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style 300 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }
    e10@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    210@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 210}"}
    style 210 fill:none,stroke:#333,stroke-width:6px

    330@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 60,\n&quot;ts&quot;: 330}"}
    style 330 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 50\n&quot;last_ts&quot;: 210,\n&quot;window_end&quot;: 300}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    space1[ ]
    style space1 fill:none,stroke:none
    330 --- space1
    linkStyle 10 stroke:none
    330 Link@== Triggers ==> Output
    linkStyle 11 stroke:#00bb00,stroke-width:3px
```

By pushing the latest timestamp beyond `300`, the tumbling window `[200, 300)` is triggered:

In [46]:
process(built_tn, customer_id=2, price=60, ts=330, w=1)

Triggers:
{'customer_id': 1, 'orders': 1, 'total_price': 50, 'last_ts': 210, 'window_end': 300}


#### Step 9

An order from customer 2 arrives late but not too late, at timestamp `120`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 330"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105 e6@-.-> 180 e7@-.-> 150 e8@-.-> 200(("200")) e9@-.-> 210 e10@-.-> 300(("300")) e11@-.-> 330
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style 300 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }
    e10@{ animate: true }
    e11@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    210@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 210}"}
    style 210 fill:none,stroke:#333,stroke-width:6px

    330@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 60,\n&quot;ts&quot;: 330}"}
    style 330 fill:none,stroke:#333,stroke-width:6px

    180@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 40,\n&quot;ts&quot;: 180}"}
    style 180 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 2,\n&quot;total_price&quot;: 90\n&quot;last_ts&quot;: 180,\n&quot;window_end&quot;: 200}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    180 Link@== Triggers ==> Output
    linkStyle 11 stroke:#00bb00,stroke-width:3px
```

This late arrival is still within the `allowed_lateness = 200`: `latest_ts = 330 - 120 = 190 < 200`. Hence, it triggers the corresponding correction of the tumbling window `[100, 200)`:

In [47]:
process(built_tn, customer_id=2, price=40, ts=180, w=1)

Triggers:
{'customer_id': 2, 'orders': 2, 'total_price': 90, 'last_ts': 180, 'window_end': 200}


#### Step 10

An order from customer 1 arrives at timestamp `510`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 510"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105 e6@-.-> 180 e7@-.-> 150 e8@-.-> 200(("200")) e9@-.-> 210 e10@-.-> 300(("300")) e11@-.-> 330 e12@-.-> 400(("400")) e13@-.-> 500(("500")) e14@-.-> 510
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style 300 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }
    e10@{ animate: true }
    e11@{ animate: true }
    e12@{ animate: true }
    e13@{ animate: true }
    e14@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    210@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 210}"}
    style 210 fill:none,stroke:#333,stroke-width:6px

    330@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 60,\n&quot;ts&quot;: 330}"}
    style 330 fill:none,stroke:#333,stroke-width:6px

    180@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 40,\n&quot;ts&quot;: 180}"}
    style 180 fill:none,stroke:#333,stroke-width:6px

    510@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 70,\n&quot;ts&quot;: 510}"}
    style 510 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 60\n&quot;last_ts&quot;: 330,\n&quot;window_end&quot;: 400}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    space1[ ]
    style space1 fill:none,stroke:none
    510 --- space1
    linkStyle 14 stroke:none
    510 Link@== Triggers ==> Output
    linkStyle 15 stroke:#00bb00,stroke-width:3px
```

This pushes the latest timestamp to `510` and closes the tumbling window `[300, 400)` (the window `[400, 500)` is empty):

In [48]:
process(built_tn, customer_id=1, price=70, ts=510, w=1)

Triggers:
{'customer_id': 2, 'orders': 1, 'total_price': 60, 'last_ts': 330, 'window_end': 400}


#### Step 11

A last order from customer 1 arrives, but this time too late at timestamp 130:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 510"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105 e6@-.-> 110 e7@-.-> 130 e8@-.-> 180 e9@-.-> 150 e10@-.-> 200(("200")) e11@-.-> 210 e12@-.-> 300(("300")) e13@-.-> 330 e14@-.-> 400(("400")) e15@-.-> 500(("500")) e16@-.-> 510
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style 300 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }
    e10@{ animate: true }
    e11@{ animate: true }
    e12@{ animate: true }
    e13@{ animate: true }
    e14@{ animate: true }
    e15@{ animate: true }
    e16@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    210@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 210}"}
    style 210 fill:none,stroke:#333,stroke-width:6px

    330@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 60,\n&quot;ts&quot;: 330}"}
    style 330 fill:none,stroke:#333,stroke-width:6px

    180@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 40,\n&quot;ts&quot;: 180}"}
    style 180 fill:none,stroke:#333,stroke-width:6px

    510@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 70,\n&quot;ts&quot;: 510}"}
    style 510 fill:none,stroke:#333,stroke-width:6px

    130@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 70,\n&quot;ts&quot;: 130}"}
    style 130 fill:none,stroke:#1565c0,stroke-width:6px
```

Why is this event "too late"? Because it does not arrive inside our `allowed_lateness`: `latest_ts = 510 - 130 = 380 > 200`. Hence, the event does not trigger a correction of tumbling window `[100, 200)` but is discarded:

In [49]:
process(built_tn, customer_id=1, price=70, ts=130, w=1)

Triggers:


<a id="hopping"></a>
### Hopping Windows

In Kafi Streams, the two operators required to set up a hopping window are `expire_hopping` and `group_by_agg_hopping`.

<a id="expire_hopping-operator"></a>
#### expire_hopping()

Syntactic sugar for `expire()` for record expiry in the context of hopping windows.

```
expire_hopping(ts_fun, size_int, hop_int, allowed_lateness_int=0, **kwargs)
```
* `ts_fun: r -> ts` timestamp function - gets an input record and returns a timestamp
* `size_int` the size (in milliseconds) of the hopping window
* `hop_int` the hop size (in milliseconds) of the hopping window
* `allowed_lateness_int` allowed lateness - adds a time (in milliseconds) to the expiry time to accommodate late arriving records

As you can see, the only difference to `expire_tumbling` is the addition of the parameter `hop_int` for specifying the hop size.

<a id="group_by_agg_hopping-operator"></a>
#### group_by_agg_hopping()

Augmented `group_by_agg()` operator for creating hopping time windows by grouping by + aggregating. Implicitly also groups by the hopping time windows and triggers the emission of aggregated hopping time windows.

```
group_by_agg_hopping(ts_fun, size_int, key_fun, agg_fun, agg_initial_any, project_fun, trigger_fun=lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1], trigger_project_fun=lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]}, trigger_positive_only_bool=True, **kwargs)
```

* `ts_fun: r -> ts`: timestamp function - gets an input record and returns a timestamp
* `size_int`: the size (in milliseconds) of the window
* `key_fun: r -> any`: the selection function for the key for the grouping
* `agg_fun: agg_any, value_any -> any`: the aggregation function; gets the result of the aggregation so far (`agg_any`) and the next selected value to aggregate (`value_any`), and returns the updated aggregation
* `agg_initial_any`: the initial value of the aggregation
* `project_fun: key_any, agg_any -> r`: the projection function; gets both the selected key and the aggregation result for that key and returns the output (=projection) of the group by + aggregation
* `trigger_fun: tuple(r, end_ts), latest_ts -> bool` the trigger function; gets a pair of a record and the window end timestamp and the latest timestamp of the input stream and returns a bool. `True` for triggering the emission of the output, `False` for not yet triggering it. Default: `lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1]` where `r_end_ts_tuple[1] = end_ts`- i.e., trigger if the latest timestamp of the input stream is greater than or equal to the window end timestamp `end_ts`.
* `trigger_project_fun: tuple(r, end_ts) -> r`: the trigger projection function; gets a pair of a record and the window end timestamp and returns a record. Default: `lambda r_end_ts_tuple: {**r_end_ts_int_tuple[0], "window_end": r_end_ts_int_tuple[1]}` i.e., add the window end timestamp to the record.
* `trigger_positive_only_bool`: trigger only updates with positive weights or also zero or negative ones. Default: `True`

This operator has the same parameters as `group_by_agg_tumbling` and also works along the same lines - only for hopping instead of tumbling windows.

You can find an example for hopping windows in the same spirit as the one above for tumbling windows here as `test_hopping()`: [test_windows.py](../../../test/streams/test_windows.py).